In [1]:
CSV_PATH    = "jobs_for_embeddings.csv"              # path to your jobs CSV file
ES_URL      = "https://localhost:9200" # Elasticsearch URL
ES_USER     = "elastic"
ES_PASS     = "75i_r1UqYAv=7u49--xR"
SLEEP_SEC   = 0.0                     # optional delay between rows (0 = max speed)
NUM_WORKERS = 8                       # threads for parallel embedding & indexing
BATCH_SIZE  = 100                     # rows per bulk-index batch


## 1 — Imports

In [2]:
import time
import json
import hashlib

import numpy as np
import pandas as pd


In [3]:
from tqdm.notebook import tqdm 

from elasticsearch import Elasticsearch

In [4]:
from sentence_transformers import SentenceTransformer 

c:\Users\Hagar Hisham Mostafa\Documents\GitHub\grad-project\.venv\Lib\site-packages\tqdm\auto.py:21: TqdmWarning: IProgress not found. Please update jupyter and ipywidgets. See https://ipywidgets.readthedocs.io/en/stable/user_install.html
  from .autonotebook import tqdm as notebook_tqdm


## 2 — Load model

In [5]:
print("Loading JobBERT — this may take a minute on first run...")
jobbert = SentenceTransformer("TechWolf/JobBERT-v2")
DIM = jobbert.get_embedding_dimension()  
EMBEDDING_DIM = DIM * 3                           
print(f"Model ready. Segment dim: {DIM} | Final embedding dim: {EMBEDDING_DIM}")

Loading JobBERT — this may take a minute on first run...


Loading weights: 100%|██████████| 199/199 [00:00<00:00, 6073.88it/s]


Model ready. Segment dim: 1024 | Final embedding dim: 3072


## 3 — Connect to Elasticsearch & create index

In [6]:
es = Elasticsearch(
    ES_URL,
    basic_auth=(ES_USER, ES_PASS),
    verify_certs=False,
    ssl_show_warn=False,
)

In [12]:
row = df.iloc[0]

embedding = generate_job_embedding(
    row["_title"],
    row["_description"],
    row["_skills"]
)

print(type(embedding))
print(len(embedding))
print(type(embedding[0]))

<class 'list'>
3072
<class 'float'>


In [14]:
action = {
    "job_id": str(row["_job_id"]),
    "normal_job_title": row["_title"],
    "job_description": row["_description"],
    "skills": row["_skills"],
    "embedding": embedding
}

resp = es.index(
    index=INDEX_NAME,
    id=str(row["_job_id"]),
    document=action
)

print(resp)

{'_index': 'jobs_strat2_normal_actual', '_id': '0', '_version': 3, 'result': 'updated', '_shards': {'total': 2, 'successful': 1, 'failed': 0}, '_seq_no': 21501, '_primary_term': 7}


In [ ]:
from elasticsearch.helpers import bulk

def flush_one(action):
    def flush_one(action):

        result = bulk(
            es,
            [action],
            raise_on_error=True,
            stats_only=False
        )

        print(f"[DEBUG] Bulk result: {result}")

        return result

    print(f"[DEBUG] Bulk result: {result}")

    return result

In [ ]:
# Delete the Elasticsearch index if it exists
INDEX_NAME = "jobs_strat2_normal"
try:
    if es.indices.exists(index=INDEX_NAME):
        resp = es.indices.delete(index=INDEX_NAME)
        print(f"Deleted index '{INDEX_NAME}': {resp}")
    else:
        print(f"Index '{INDEX_NAME}' does not exist.")
except Exception as e:
    print(f"Error deleting index '{INDEX_NAME}': {e}")

In [ ]:
INDEX_NAME = "jobs_strat2_normal_actual"

mapping = {
    "mappings": {
        "properties": {
            "job_id":          {"type": "keyword"},
            "normal_job_title":       {"type": "text"},
            "job_description": {"type": "text"},
            "skills":          {"type": "keyword"},
            "embedding": {
                "type":       "dense_vector",
                "dims":       EMBEDDING_DIM,
                "index":      True,
                "similarity": "cosine",
            },
        }
    }
}

try:
    exists = es.indices.exists(index=INDEX_NAME)
except Exception as e:
    print(f"Error checking index existence: {e}")
    if getattr(e, "status_code", None) == 401:
        print("Authentication failed (401). Check ES_URL, ES_USER and ES_PASS.")
    raise

if not exists:
    try:
        resp = es.indices.create(index=INDEX_NAME, body=mapping)
        print(f"Created index '{INDEX_NAME}': {resp}")
    except Exception as e:
        print(f"Error creating index '{INDEX_NAME}': {e}")
        raise
else:
    print(f"Index '{INDEX_NAME}' already exists — skipping creation")

In [7]:
# Check whether the Elasticsearch index exists and whether it's empty
INDEX_NAME = "jobs_strat2_normal_actual"

try:
    if not es.indices.exists(index=INDEX_NAME):
        print(f"Index '{INDEX_NAME}' does not exist.")
    else:
        count = es.count(index=INDEX_NAME)["count"]
        if count == 0:
            print(f"Index '{INDEX_NAME}' is empty (0 documents).")
        else:
            print(f"Index '{INDEX_NAME}' contains {count:,} documents.")
except Exception as e:
    print("Error checking index:", str(e))

Index 'jobs_strat2_normal_actual' contains 21,500 documents.


## 4 — Load & preview CSV

In [8]:
df = pd.read_csv(CSV_PATH)
print(f"Loaded {len(df):,} rows")


Loaded 213,848 rows


## 5 — Clean & prepare

In [11]:
def generate_job_embedding(title: str, description: str, skills: list[str]) -> list[float]:
    """
    Concatenate JobBERT encodings of title, description, and skills
    into a single 3072-dim vector: [title(1024) | desc(1024) | skills(1024)]
    """
    title_emb  = jobbert.encode(title       if title       else "unknown")
    desc_emb   = jobbert.encode(description if description else "")
    skills_emb = jobbert.encode(" ".join(skills) if skills else "general")
    return np.concatenate([title_emb, desc_emb, skills_emb]).tolist()


# Apply cleaning
df["_job_id"] = df["job_id"]
df["_title"]   = df["esco_title"]
df["_skills"]  = df["new_skills"].apply(lambda x: [s.strip() for s in str(x).split("|") if s.strip()])
df["_description"] = df["job_description"]

before = len(df)
df = df[df["_title"] != ""].reset_index(drop=True)
print(f"Dropped {before - len(df)} rows with no title. Ready to embed: {len(df):,}")

Dropped 0 rows with no title. Ready to embed: 213,848


## 6 — Dry run (first 3 rows)

In [ ]:
for _, row in df.head(3).iterrows():
    emb = generate_job_embedding(row["_title"], row["_description"], row["_skills"])
    print(f"[OK] '{row['_title']}' → {len(emb)} dims | skills: {row['_skills'][:3]}")

## 7 — Bulk embed & index
Embeds and indexes every row directly into Elasticsearch. Errors are collected without stopping the run.

In [ ]:
from concurrent.futures import ThreadPoolExecutor, as_completed
from elasticsearch.helpers import bulk
import threading
import traceback
import time

# =========================
# CONFIG
# =========================

EMBED_BATCH_SIZE = 500
ES_CHUNK_SIZE = 500
MAX_RETRIES = 3

# =========================
# STATE
# =========================

_lock = threading.Lock()

embedded_count = 0
indexed_count = 0
errors = []

# =========================
# GET EXPECTED VECTOR DIMS
# =========================

mapping = es.indices.get_mapping(index=INDEX_NAME)

try:
    expected_dims = (
        mapping[INDEX_NAME]["mappings"]["properties"]["embedding"]["dims"]
    )
    print(f"[INFO] ES expects embedding dims = {expected_dims}")
except Exception:
    expected_dims = None
    print("[WARNING] Could not determine embedding dimensions from mapping")

# =========================
# FIND EXISTING DOCUMENTS
# =========================

all_job_ids = [
    str(row["_job_id"])
    for _, row in df.iterrows()
    if row["_job_id"] not in (None, "")
]

existing_ids = set()

for i in range(0, len(all_job_ids), ES_CHUNK_SIZE):
    chunk = all_job_ids[i:i + ES_CHUNK_SIZE]

    try:
        resp = es.mget(index=INDEX_NAME, body={"ids": chunk})

        for doc in resp.get("docs", []):
            if doc.get("found"):
                existing_ids.add(str(doc["_id"]))

    except Exception as e:
        print(f"[WARNING] mget failed: {e}")

rows_to_index = [
    row
    for _, row in df.iterrows()
    if str(row["_job_id"]) not in existing_ids
]

print(
    f"[INFO] Total={len(df):,} | "
    f"Existing={len(existing_ids):,} | "
    f"To index={len(rows_to_index):,}"
)

if not rows_to_index:
    print("Nothing to index.")
    raise SystemExit

# =========================
# EMBEDDING FUNCTION
# =========================

def embed_and_prepare(row):

    global embedded_count

    job_id = str(row["_job_id"])

    try:

        start = time.time()

        embedding = generate_job_embedding(
            row["_title"],
            row["_description"],
            row["_skills"]
        )

        if embedding is None:
            raise ValueError("Embedding is None")

        if len(embedding) == 0:
            raise ValueError("Embedding is empty")

        if expected_dims and len(embedding) != expected_dims:
            raise ValueError(
                f"Dimension mismatch. "
                f"Expected={expected_dims}, "
                f"Got={len(embedding)}"
            )

        elapsed = time.time() - start

        with _lock:
            embedded_count += 1

        print(
            f"[EMBEDDED] {job_id} "
            f"({len(embedding)} dims, {elapsed:.2f}s)"
        )

        return {
            "_index": INDEX_NAME,
            "_id": job_id,
            "_source": {
                "job_id": job_id,
                "normal_job_title": row["_title"],
                "job_description": row["_description"],
                "skills": row["_skills"],
                "embedding": embedding,
            },
        }

    except Exception as e:

        err = {
            "job_id": job_id,
            "stage": "embedding",
            "error": str(e),
            "trace": traceback.format_exc(),
        }

        with _lock:
            errors.append(err)

        print(f"[EMBED ERROR] {job_id}: {e}")

        return None

# =========================
# BULK INDEX FUNCTION
# =========================

def bulk_index(actions):

    global indexed_count

    if not actions:
        return

    for attempt in range(MAX_RETRIES):

        try:

            success, failed = bulk(
                es,
                actions,
                raise_on_error=False,
                stats_only=False,
                chunk_size=min(len(actions), 500)
            )

            indexed_count += success

            print(
                f"[INDEX] Success={success:,} "
                f"Failed={len(failed):,}"
            )

            if failed:

                for f in failed:

                    errors.append(
                        {
                            "stage": "indexing",
                            "error": str(f)
                        }
                    )

                    print(f"[INDEX ERROR] {f}")

            return

        except Exception as e:

            print(
                f"[RETRY {attempt+1}/{MAX_RETRIES}] "
                f"Bulk failed: {e}"
            )

            time.sleep(2)

    errors.append(
        {
            "stage": "bulk",
            "error": "Max retries exceeded"
        }
    )

# =========================
# PARALLEL EMBEDDING
# =========================

actions_buffer = []

with ThreadPoolExecutor(max_workers=NUM_WORKERS) as pool:

    futures = {
        pool.submit(embed_and_prepare, row): row["_job_id"]
        for row in rows_to_index
    }

    print(
        f"[INFO] Submitted "
        f"{len(futures):,} embedding jobs"
    )

    for future in tqdm(
        as_completed(futures),
        total=len(futures),
        desc="Embedding"
    ):

        action = future.result()

        if action is None:
            continue

        actions_buffer.append(action)

        if len(actions_buffer) >= EMBED_BATCH_SIZE:

            bulk_index(actions_buffer)

            actions_buffer.clear()

# =========================
# FINAL FLUSH
# =========================

if actions_buffer:
    bulk_index(actions_buffer)

# =========================
# REFRESH INDEX
# =========================

try:
    es.indices.refresh(index=INDEX_NAME)
except Exception as e:
    print(f"[WARNING] Refresh failed: {e}")

# =========================
# VERIFY
# =========================

try:
    count = es.count(index=INDEX_NAME)["count"]
    print(f"[VERIFY] Documents currently in ES: {count:,}")
except Exception as e:
    print(f"[VERIFY ERROR] {e}")

# =========================
# SUMMARY
# =========================

print("\n========== SUMMARY ==========")
print(f"Embedded: {embedded_count:,}")
print(f"Indexed : {indexed_count:,}")
print(f"Errors  : {len(errors):,}")

if errors:
    print("\nSample Errors:")
    for err in errors[:10]:
        print(err)

[INFO] ES expects embedding dims = 3072
[INFO] Total=213,848 | Existing=21,500 | To index=192,348
[INFO] Submitted 192,348 embedding jobs
[EMBEDDED] 21507 (3072 dims, 14.29s)
[EMBEDDED] 21505 (3072 dims, 14.75s)
[EMBEDDED] 21501 (3072 dims, 14.81s)
[EMBEDDED] 21504 (3072 dims, 14.85s)
[EMBEDDED] 21506 (3072 dims, 14.88s)
[EMBEDDED] 21503 (3072 dims, 14.88s)
[EMBEDDED] 21500 (3072 dims, 14.90s)
[EMBEDDED] 21502 (3072 dims, 15.26s)
[EMBEDDED] 21508 (3072 dims, 3.12s)
[EMBEDDED] 21511 (3072 dims, 4.46s)
[EMBEDDED] 21510 (3072 dims, 4.56s)
[EMBEDDED] 21509 (3072 dims, 4.97s)
[EMBEDDED] 21512 (3072 dims, 4.93s)
[EMBEDDED] 21514 (3072 dims, 5.10s)
[EMBEDDED] 21515 (3072 dims, 4.88s)
[EMBEDDED] 21516 (3072 dims, 4.25s)
[EMBEDDED] 21513 (3072 dims, 7.16s)
[EMBEDDED] 21523 (3072 dims, 2.71s)
[EMBEDDED] 21518 (3072 dims, 5.14s)
[EMBEDDED] 21517 (3072 dims, 5.39s)
[EMBEDDED] 21519 (3072 dims, 5.37s)
[EMBEDDED] 21520 (3072 dims, 5.40s)
[EMBEDDED] 21522 (3072 dims, 5.13s)
[EMBEDDED] 21521 (3072 dim

## 8 — Inspect errors

In [ ]:
if errors:
    err_df = pd.DataFrame(errors)
    display(err_df)
    err_df.to_csv("embedding_errors.csv", index=False)
    print("Saved to embedding_errors.csv")
else:
    print("No errors — all jobs indexed successfully.")

## 9 — Verify: kNN search test

In [ ]:
# Build a query embedding from the first row's title + skills
sample = df.iloc[0]

query_vector = np.concatenate([
    jobbert.encode(sample["_title"]),
    np.zeros(DIM),                                                         # zero-pad description slot
    jobbert.encode(" ".join(sample["_skills"]) if sample["_skills"] else "general"),
]).tolist()

response = es.search(
    index=INDEX_NAME,
    body={
        "knn": {
            "field":          "embedding",
            "query_vector":   query_vector,
            "k":              5,
            "num_candidates": 50,
        },
        "_source": ["job_id", "job_title", "skills"],
    },
)

print(f"Top 5 matches for '{sample['_title']}':")
for hit in response["hits"]["hits"]:
    src = hit["_source"]
    print(f"  [{hit['_score']:.4f}] {src['job_title']}")

## 10 — Index stats

In [ ]:

count = es.count(index=INDEX_NAME)["count"]
print(f"Total documents in '{INDEX_NAME}': {count}")